In [1]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import pickle
import joblib

PROJECT_ROOT = Path('/Users/Lekshmi Priya/OneDrive/Documents/GitHub/Credit-Card-Fraud-Detection-System')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import *
from src.utils import get_logger, set_plot_style, format_number, format_pct
from src.data_loader import load_raw_data, get_dataset_summary
from src.preprocessing import (
    handle_missing_values,
    cap_outliers,
    encode_categoricals,
    split_data,
    scale_features,
    run_preprocessing_pipeline,
)

logger = get_logger('Preprocessing-EDA')
set_plot_style()

RANDOM_SEED = RANDOM_STATE
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

SAMPLE_FRAC = 0.3
OUTPUT_DIR = Path(PROJECT_ROOT) / 'outputs'
MODEL_OUT_DIR = Path(PROJECT_ROOT) / 'models'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Output dir:   {OUTPUT_DIR}')
print(f'Sample frac:  {SAMPLE_FRAC}')
print(f'Random seed:  {RANDOM_SEED}')

ModuleNotFoundError: No module named 'torch_geometric'

In [2]:
import torch

def get_torch_versions():
    """Gets the versions of torch and cuda."""
    try:
        torch_version = torch.__version__
        cuda_version = torch.version.cuda
        if cuda_version:
            cuda = "cu" + cuda_version.replace(".", "")
        else:
            cuda = "cpu"
        return torch_version, cuda
    except Exception:
        return None, None

torch_version, cuda = get_torch_versions()

if torch_version:
    # Construct the installation command
    command = f"pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-{torch_version}+{cuda}.html"
    
    print(f"Running installation command:\n{command}")
    
    # Execute the command
    import subprocess
    import sys
    
    result = subprocess.run(command.split(), capture_output=True, text=True)
    
    if result.returncode == 0:
        print("Installation successful!")
        print(result.stdout)
    else:
        print("Installation failed.")
        print(result.stderr)
else:
    print("Could not determine PyTorch version. Please install torch_geometric manually.")


Running installation command:
pip install torch-scatter torch-sparse torch-cluster torch-spline-conv torch-geometric -f https://data.pyg.org/whl/torch-2.11.0+cpu+cpu.html
Installation failed.
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "C:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "C:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_s

In [3]:
import subprocess
import sys

def install_package(package):
    print(f"Installing {package}...")
    result = subprocess.run([sys.executable, "-m", "pip", "install", package], capture_output=True, text=True)
    if result.returncode == 0:
        print(f"Successfully installed {package}")
        print(result.stdout)
    else:
        print(f"Failed to install {package}")
        print(result.stderr)
        return False
    return True

packages = [
    "torch-scatter",
    "torch-sparse",
    "torch-cluster",
    "torch-spline-conv",
    "torch-geometric"
]

for pkg in packages:
    if not install_package(pkg):
        print(f"Stopping installation due to error with {pkg}.")
        break

Installing torch-scatter...
Failed to install torch-scatter
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "c:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "c:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\ProgramData\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\Lekshmi Priya\AppData\Local\Temp\pip-build-env-k33gk9jt\overlay\Lib\site-packages\setuptools\build_m